**Shared experiment for Problems 6 and 7**

The two AI problems share one classification experiment. The new task is to  **estimate parameters from finite samples, train models, select hyperparameters, and evaluate on independent data.**

Use the generator below. Observations are independent, and the three datasets are mutually independent. Do not add standardization or PCA in this edition.

In [86]:
import numpy as np

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    y = rng.binomial(1, 0.4, n)
    means = np.array([[-1., 0.], [1., 0.5]])
    cov = np.array([[1., 0.4], [0.4, 1.5]])
    X = rng.multivariate_normal(np.zeros(2), cov, n)
    return X + means[y], y

X_train, y_train = make_data(120, 20261102)
X_valid, y_valid = make_data(120, 20261103)
# Generate and use only after model selection in Problem 7:
X_test, y_test = make_data(2000, 20261104)

No.6. Implement `fit_gda(X, y, gamma) -> theta`, returning $\theta=(b,w)\in\mathbb R^{d+1}$ so that the predicted log-odds are $b+x^\top w.$ Solve a linear system instead of explicitly forming an inverse. Check the following data:

```
X = np.array([[-2.,0.], [0.,0.], [0.,1.], [2.,1.]])
y = np.array([0,0,1,1])
```

For $\gamma=0.5$, obtain $\theta=(-1,4/3,2)$ to within $10^{-10}$. For $\gamma=0$, detect the singularity.

In [88]:
# We will return the (b, w) \log\frac{P(Y=1\mid X=x)}{P(Y=0\mid X=x)} = b + x^\top w}
# Apply the explicit formula
def fit_gda(X, y, gamma): # Gaussian Discriminant Analysis
    n, d, n0, n1 = X.shape[0], X.shape[1], 0, 0
    mu0 = np.zeros((1, d))
    mu1 = np.zeros((1, d))
    for i in range(n):
        if y[i] == 0:
            mu0 = mu0 + X[i]
            n0 = n0 + 1
        else:
            mu1 = mu1 + X[i]
            n1 = n1 + 1
    mu0, mu1 = mu0/n0, mu1/n1
    Sigma = gamma * np.identity(d) # We use regularized \hat\Sigma_\gamma
    for i in range(n):
        if y[i] == 0:
            Sigma = Sigma + (X[i] - mu0).T @ (X[i] - mu0)
        else:
            Sigma = Sigma + (X[i] - mu1).T @ (X[i] - mu1)
    is_singular = np.linalg.matrix_rank(Sigma) < min(Sigma.shape)
    if is_singular:
        print("Σ is singular.")
        return
    Sigma = Sigma/n
    w = np.linalg.solve(Sigma, (mu1 - mu0).T)
    b = -w.T @ (mu0 + mu1).T/2 + np.log(n1) - np.log(n0) # \log(\pi/(1-\pi))
    theta = np.append(b, w)
    return theta

On the main dataset, select from $\gamma\in\{0,0.01,0.1\}$ using validation **mean log loss**, breaking ties in favor of the larger $\gamma.$ Use first the initial 30 training observations and then all 120. Record validation error and log loss. Keep the 120-observation model for final evaluation; do not inspect the test set yet.

In [90]:
def valid(X, y, theta):
    e, l = 0, 0
    for i in range(X.shape[0]):
        v = np.append(1, X[i])
        r = v @ theta
        pi = 1/(1 + np.exp(-r)) # Sigmoid(b + x^\top w)
        l = l - (y[i] * np.log(pi) + (1-y[i]) * np.log(1 - pi)) # add the p0 if y_i = 0, add the p1 if y_i = 1.
        if (r >= 0 and y[i] == 0) or (r < 0 and y[i] == 1):
            e = e + 1
    return e/X.shape[0], l/X.shape[0]

# initial 30 training observations
X, y = X_train[:30], y_train[:30]
theta1 = fit_gda(X, y, 0)
theta2 = fit_gda(X, y, 0.01)
theta3 = fit_gda(X, y, 0.1)
Xt, yt = X_valid, y_valid
print(valid(Xt, yt, theta1))
print(valid(Xt, yt, theta2))
print(valid(Xt, yt, theta3))

print('\n')

# all 120 training observations
X, y = X_train, y_train
theta1 = fit_gda(X, y, 0)
theta2 = fit_gda(X, y, 0.01)
theta3 = fit_gda(X, y, 0.1)
print(valid(Xt, yt, theta1))
print(valid(Xt, yt, theta2))
print(valid(Xt, yt, theta3))

(0.175, 0.39247054037908835)
(0.175, 0.3924738427481132)
(0.175, 0.39250474648034844)


(0.15, 0.3593847601374091)
(0.15, 0.3593875722619396)
(0.15, 0.3594129922972805)


No.7. Implement `loss_grad(theta, Z, y, lam)`. Use the first 8 training rows, $\theta=(0.2,-0.3,0.4), \lambda=0.1,$ and centered finite differences with step $h=10^{-6}$. Verify $$\frac{\|g-g_{\rm num}\|_2}
{\max(1,\|g\|_2+\|g_{\rm num}\|_2)}<10^{-7}.$$
Also check that the loss and gradient remain finite for scores of $\pm1000$.

In [92]:
def loss_grad(theta, Z, y, lam):
    n, d = Z.shape[0], Z.shape[1]-1
    l, grad = 0, np.zeros(d+1)
    for i in range(n):
        mi = max(-y[i] * (Z[i].T @ theta), (1-y[i]) * (Z[i].T @ theta))
        l = l + np.logaddexp(-y[i] * (Z[i].T @ theta) - mi, (1-y[i]) * (Z[i].T @ theta) - mi) + mi
        if -Z[i].T @ theta > 0:
            pi = 1/(1 + np.exp(Z[i].T @ theta))
            grad = grad + (1 - pi - y[i]) * Z[i]
        else:
            pi = 1/(1 + np.exp(-Z[i].T @ theta))
            grad = grad + (pi - y[i]) * Z[i]
    P = np.eye(d + 1)
    P[0, 0] = 0
    l = l/n + lam * 0.5 * theta.T @ P @ theta
    grad = grad/n + lam * P @ theta
    return l, grad

# ---------- Test Data ------------

X8 = X_train[:8]
y = y_train[:8]

Z = np.column_stack((np.ones(8), X8))

theta = np.array([0.2, -0.3, 0.4])
lam = 0.1

loss, grad = loss_grad(
    theta, Z, y, lam
)

print("Loss:", loss)
print("Gradient:", grad)
print("Finite loss:", np.isfinite(loss))
print("Finite gradient:", np.all(np.isfinite(grad)))


# Finite Difference Check
h = 1e-6
n, d = Z.shape[0], Z.shape[1]-1
g_num = np.zeros(d+1)
for i in range(d+1):
    e = np.zeros(d+1)
    e[i] = 1
    l1, l2 = loss_grad(theta + h*e, Z, y, lam)[0], loss_grad(theta - h*e, Z, y, lam)[0]
    g_num[i] = (l1 - l2)/(2*h)
g = loss_grad(theta, Z, y, lam)[1]
E = ((g - g_num).T @ (g - g_num)) ** 0.5 / max((g.T @ g)**0.5 + (g_num.T @ g_num)**0.5, 1)
print("Finite Difference Check:", E < 1e-7)

# Extreme theta for stableness
for score in [1000.0, -1000.0]:
    theta_extreme = np.array([score, 0.0, 0.0])
    loss, g = loss_grad(theta_extreme, Z, y, lam)

    print(f"score = {score}: loss = {loss}, gradient = {g}")
    assert np.isfinite(loss)
    assert np.all(np.isfinite(g))

Loss: 0.7193186439900554
Gradient: [ 0.09304892 -0.55236961 -0.27743007]
Finite loss: True
Finite gradient: True
Finite Difference Check: True
score = 1000.0: loss = 500.0, gradient = [ 0.5        -0.44933223 -0.15385718]
score = -1000.0: loss = 500.0, gradient = [-0.5        -0.610241   -0.75913434]


Compare full-batch gradient descent with SGD using batches of 20. At every SGD step, sample indices uniformly **with replacement**, independently of previous steps. Prove conditional unbiasedness of its gradient. Each candidate model processes **24,000 observations**: 200 GD steps or 1,200 SGD steps. Initialize both methods at zero and use
$$L=\frac{\|Z\|_2^2}{4n}+\lambda,\qquad
\eta_{\rm GD}=\frac1L,\quad
\eta_t^{\rm SGD}=\frac1{L(1+t/200)}.$$
Prove that $L$ is an upper bound on the gradient's Lipschitz constant. Record the full training objective after every 2,400 processed observations.

In [94]:
def grad_des(X, y, lam, repeats, theta0): # standard GD
    n, d = X.shape
    Z = np.column_stack((np.ones(n), X))
    theta = theta0
    L = np.linalg.norm(Z, ord=2)**2/(4*n) + lam 
    for i in range(repeats):
        grad = loss_grad(theta, Z, y, lam)[1]
        theta = theta - grad/L
    return theta

def Sgrad_des(X, y, lam, repeats, theta0): # SGD
    n, d = X.shape
    Z = np.column_stack((np.ones(n), X))
    theta = theta0
    L = np.linalg.norm(Z, ord=2)**2/(4*n) + lam
    for t in range(repeats):
        rng = np.random.default_rng()
        idx = rng.choice(n, size=20, replace=True)
        grad = np.zeros(d+1)
        for i in idx:
            if -Z[i].T @ theta > 0:
                pi = 1/(1 + np.exp(Z[i].T @ theta))
                grad = grad + (1 - pi - y[i]) * Z[i]
            else:
                pi = 1/(1 + np.exp(-Z[i].T @ theta))
                grad = grad + (pi - y[i]) * Z[i]
        P = np.eye(d + 1)
        P[0, 0] = 0
        grad = grad/20 + lam * P @ theta
        theta = theta - grad/(L*(1 + t/200))
    return theta

In [95]:
def validation_log_loss(theta, X, y):
    Z = np.column_stack((np.ones(len(y)), X))
    scores = Z @ theta
    return np.mean(np.logaddexp(0.0, (1 - 2 * y) * scores))


lambdas = [0.001, 0.01, 0.1]
results = {}

for name, train_fn, steps in [
    ("GD", grad_des, 200),
    ("SGD", Sgrad_des, 1200),
]:
    candidates = []

    for lam in lambdas:
        theta0 = np.zeros(X_train.shape[1] + 1)

        theta = train_fn(
            X_train, y_train,
            lam=lam,
            repeats=steps,
            theta0=theta0,
        )

        val_loss = validation_log_loss(theta, X_valid, y_valid)
        assert np.isfinite(val_loss), f"{name}, lambda={lam}: 验证损失非有限"

        candidates.append({
            "lam": lam,
            "theta": theta.copy(),
            "val_loss": float(val_loss),
        })

        print(f"{name:3s} | lambda={lam:.3f} | "
              f"validation log loss={val_loss:.10f}")

    best = min(candidates, key=lambda c: (c["val_loss"], -c["lam"]))
    results[name] = best

    print(f"选中 {name}: lambda={best['lam']}, "
          f"validation log loss={best['val_loss']:.10f}\n")

theta_GD = results["GD"]["theta"].copy()
lambda_GD = results["GD"]["lam"]

theta_SGD = results["SGD"]["theta"].copy()
lambda_SGD = results["SGD"]["lam"]

print("最终 GD 参数:", theta_GD)
print("最终 SGD 参数:", theta_SGD)

GD  | lambda=0.001 | validation log loss=0.3640256545
GD  | lambda=0.010 | validation log loss=0.3708099783
GD  | lambda=0.100 | validation log loss=0.4221133446
选中 GD: lambda=0.001, validation log loss=0.3640256545

SGD | lambda=0.001 | validation log loss=0.3597625791
SGD | lambda=0.010 | validation log loss=0.3712321542
SGD | lambda=0.100 | validation log loss=0.4224882430
选中 SGD: lambda=0.001, validation log loss=0.3597625791

最终 GD 参数: [-0.55714731  2.14696311 -0.10265828]
最终 SGD 参数: [-0.50221041  2.20629608 -0.18531622]


In [104]:
# We campare the 3 algorithms: GD, SGD, GDA(Gaussian Discriminant Analysis

gamma_GDA = 0.0
theta_GDA = np.array([
    -0.502171461486, 2.371218453433, -0.208630192976
])

lambda_GD = 0.001
theta_GD = np.array([
    -0.557147312763, 2.146963114359, -0.102658277865
])

lambda_SGD = 0.001
theta_SGD = np.array([
    -0.523321021215, 2.053339591735, -0.111955132107
])

for name, theta in [("GDA", theta_GDA), ("GD", theta_GD), ("SGD", theta_SGD)]:
    error, loss = valid(X_test, y_test, theta)

    n = len(y_test)
    z = 1.959963984540054
    denominator = 1 + z**2 / n
    center = (error + z**2 / (2*n)) / denominator
    radius = z * np.sqrt(error * (1-error)/n + z**2/(4*n**2)) / denominator

    print(
        f"{name}: error rate={error:.4%}, "
        f"mean log loss={loss:.6f}, "
        f"Wilson interval=({center-radius:.4%}, {center+radius:.4%})"
    )

GDA: error rate=15.3000%, mean log loss=0.345202, Wilson interval=(13.7889%, 16.9441%)
GD: error rate=15.2000%, mean log loss=0.344893, Wilson interval=(13.6934%, 16.8401%)
SGD: error rate=15.1000%, mean log loss=0.344233, Wilson interval=(13.5978%, 16.7360%)
